# 02 — Climate Processing (ERA5 → NUTS2)

## Objective

Process ERA5 climate data and aggregate it to NUTS2 regions (2010–2023).

---

## Data

- ERA5 (Copernicus):
  - 2m temperature  
  - Total precipitation  

---

## Method

- Download monthly ERA5 data  
- Convert to annual values  
- Spatial aggregation:
  - Grid → NUTS2 regions (spatial join)  

Variables constructed:
- Average temperature (°C)  
- Average precipitation (mm)  

---

## Output

Climate dataset:

- NUTS2 regions  
- Annual observations (2010–2023)  

Saved as:

`data/raw/climate_nuts2.csv`

---

In [2]:
# %%
import pandas as pd
import geopandas as gpd
import xarray as xr
import cdsapi

In [3]:
# -------------------------------
# 1. DOWNLOAD ERA5
# -------------------------------
client = cdsapi.Client(
    url="https://cds.climate.copernicus.eu/api",
    key="e37d8887-d2eb-4896-a3b9-113fa5dfac36"
)

dataset = "reanalysis-era5-single-levels-monthly-means"

request = {
    "product_type": "monthly_averaged_reanalysis",
    "variable": ["2m_temperature", "total_precipitation"],
    "year": [str(y) for y in range(2010, 2024)],
    "month": ["01","02","03","04","05","06","07","08","09","10","11","12"],
    "time": "00:00",
    "area": [72, -25, 34, 45],
    "format": "netcdf"
}

file_path = "../data/raw/era5_2010_2023.nc"

client.retrieve(dataset, request).download(file_path)

print("Download OK")

2026-05-03 16:23:03,607 INFO Request ID is 00f44ae9-82e3-41a4-bbe7-347b11f7aa4c
2026-05-03 16:23:03,673 INFO status has been updated to accepted
2026-05-03 16:23:17,117 INFO status has been updated to successful


287f6a30733f9be46cbdf7da57902605.zip:   0%|          | 0.00/20.6M [00:00<?, ?B/s]

Download OK


In [5]:
# -------------------------------
# 2. LOAD DATA (ROBUST)
# -------------------------------
import os
import zipfile
import xarray as xr

def load_era5(file_path):

    # --- check file exists
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"{file_path} not found")

    # --- detect ZIP
    if zipfile.is_zipfile(file_path):
        print("Detected ZIP → extracting")

        extract_dir = file_path + "_extracted"
        os.makedirs(extract_dir, exist_ok=True)

        with zipfile.ZipFile(file_path, 'r') as z:
            z.extractall(extract_dir)

        files = [
            os.path.join(extract_dir, f)
            for f in os.listdir(extract_dir)
            if f.endswith(".nc")
        ]

        if len(files) == 0:
            raise ValueError("No NetCDF files found after extraction")

        print(f"{len(files)} NetCDF files found")

        ds_list = [xr.open_dataset(f) for f in files]
        ds = xr.merge(ds_list)

    else:
        print("Detected NetCDF → loading directly")
        ds = xr.open_dataset(file_path, engine="netcdf4")

    return ds


ds = load_era5(file_path)

Detected ZIP → extracting
2 NetCDF files found


C:\Users\henri_ugzoq54\AppData\Local\Temp\ipykernel_8672\1937175581.py:36: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'valid_time' ('valid_time',) The recommendation is to set join explicitly for this case.
  ds = xr.merge(ds_list)
C:\Users\henri_ugzoq54\AppData\Local\Temp\ipykernel_8672\1937175581.py:36: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds = xr.merge(ds_list)
C:\Users\henri_ugzoq54\AppData\Local\Temp\ipyker

In [6]:
# -------------------------------
# 3. TO DATAFRAME
# -------------------------------
df = ds.to_dataframe().reset_index()

if "valid_time" in df.columns:
    df = df.rename(columns={"valid_time": "time"})

df = df.rename(columns={
    "t2m": "temperature",
    "tp": "precipitation"
})

df["temperature"] -= 273.15
df["precipitation"] *= 1000
df["year"] = df["time"].dt.year

# -------------------------------
# 4. GRID AGGREGATION
# -------------------------------
df = df.groupby(["latitude","longitude","year"]).agg({
    "temperature": "mean",
    "precipitation": "sum"
}).reset_index()

# -------------------------------
# 5. LOAD NUTS2
# -------------------------------
nuts = gpd.read_file(
    "https://gisco-services.ec.europa.eu/distribution/v2/nuts/shp/NUTS_RG_01M_2021_4326.shp.zip"
)

nuts2 = nuts[nuts["LEVL_CODE"] == 2][["NUTS_ID","geometry"]]
nuts2 = nuts2.rename(columns={"NUTS_ID": "region"})
nuts2 = nuts2.set_crs("EPSG:4326", allow_override=True)

# -------------------------------
# 6. SPATIAL JOIN
# -------------------------------
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326"
)

joined = gpd.sjoin(gdf, nuts2, how="inner", predicate="intersects")

# -------------------------------
# 7. FINAL AGGREGATION
# -------------------------------
climate_nuts2 = (
    joined.groupby(["region","year"])
    .agg({
        "temperature": "mean",
        "precipitation": "sum"
    })
    .reset_index()
)

# -------------------------------
# 8. CHECK
# -------------------------------
print("Climate NUTS2:", climate_nuts2.shape)
print("Regions:", climate_nuts2["region"].nunique())
print("Years:", climate_nuts2["year"].min(), "-", climate_nuts2["year"].max())
print("Missing:\n", climate_nuts2.isna().mean())

# -------------------------------
# 9. SAVE
# -------------------------------
climate_nuts2.to_csv("../data/raw/climate_nuts2.csv", index=False)

print("Saved climate data:", climate_nuts2.shape)

Climate NUTS2: (4438, 4)
Regions: 317
Years: 2010 - 2023
Missing:
 region           0.0
year             0.0
temperature      0.0
precipitation    0.0
dtype: float64
Saved climate data: (4438, 4)


In [7]:
print(climate_nuts2.shape)
print(climate_nuts2.head())

print("Regions:", climate_nuts2["region"].nunique())
print("Years:", climate_nuts2["year"].unique())

print("Missing:\n", climate_nuts2.isna().mean())

(4438, 4)
  region  year  temperature  precipitation
0   AL01  2010    11.115707    1355.863525
1   AL01  2011    10.968773     586.859680
2   AL01  2012    11.402008     927.210815
3   AL01  2013    11.517006    1002.031311
4   AL01  2014    11.653899     974.426270
Regions: 317
Years: [2010 2011 2012 2013 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023]
Missing:
 region           0.0
year             0.0
temperature      0.0
precipitation    0.0
dtype: float64
